## モル分率により重み付けを行いprocessed_data.csvを作成

### step1:各行ごとにmolを計算

In [1]:
import pandas as pd

df = pd.read_csv(r"../../data/interim/merged_basic_descriptors.csv")

# 念のため数値化（文字列混入対策）
mc = pd.to_numeric(df["material_concentration"], errors="coerce")
prop = pd.to_numeric(df["proportion"], errors="coerce") / 100.0  # % -> 比率
mw = pd.to_numeric(df["ExactMolWt"], errors="coerce")

mol = mc * prop / mw

insert_at = df.columns.get_loc("proportion") + 1
df.insert(insert_at, "mol", mol)

df_step1 = df

### step2:composition_idごとにtotal_molを計算

In [2]:
df_step2 = df_step1.copy()

total_mol = df_step2.groupby("composition_id")["mol"].transform("sum")

if "total_mol" in df_step2.columns:
    df_step2.drop(columns="total_mol", inplace=True)

df_step2.insert(df_step2.columns.get_loc("mol") + 1, "total_mol", total_mol)

### step3:各行ごとにmol_fractionを計算

In [3]:
df_step3 = df_step2.copy()

bad = df_step3["total_mol"].isna() | (df_step3["total_mol"] == 0)
if bad.any():
    raise ValueError(f"total_mol が 0 または NaN の行があります: {bad.sum()} 行")

mol_fraction = df_step3["mol"] / df_step3["total_mol"]

if "mol_fraction" in df_step3.columns:
    df_step3.drop(columns="mol_fraction", inplace=True)

df_step3.insert(df_step3.columns.get_loc("total_mol") + 1, "mol_fraction", mol_fraction)

### step4:各行ごとに記述子をmol_fractionで重み付けしてweighted_記述子列を作成

In [4]:
df_step4 = df_step3.copy()

# 重み付け対象（元のRDKit記述子列のみ：std_/range_/max_/min_ は除外）
exclude = {
    "id", "experiment_id", "composition_id", "material_id", "structure_id",
    "material_concentration", "proportion", "mol", "total_mol", "mol_fraction",
    "humidity", "temperature", "wvp",
}
base_desc_cols = df_step4.select_dtypes("number").columns.difference(exclude)

w = df_step4[base_desc_cols].fillna(0.0).mul(df_step4["mol_fraction"], axis=0)
w.columns = ["weighted_" + c for c in base_desc_cols]

# 再実行対策（既存 weighted_ を消してから追加）
df_step4 = df_step4.loc[:, ~df_step4.columns.str.startswith("weighted_")]
df_step4 = pd.concat([df_step4, w], axis=1)

### step5:composition_idごとに各記述子の分散（std）、範囲（range）、最大・最小値（max/min）を計算し、その分、列を追加=>この処理をスキップ

In [5]:
df_step5 = df_step4.copy()

### step6:composition_idごとに足して１つの行にする

In [6]:
df_step6_src = df_step5.copy()

drop_cols = [
    "id", "material_id", "name", "structure_id", "material_concentration",
    "smiles", "proportion", "mol", "mol_fraction", "total_mol",
] + list(base_desc_cols)

df_step6_src = df_step6_src.drop(columns=[c for c in drop_cols if c in df_step6_src.columns])

weighted_cols = [c for c in df_step6_src.columns if c.startswith("weighted_")]
other_cols = [c for c in df_step6_src.columns if c not in (["composition_id"] + weighted_cols)]

df_step6 = (
    df_step6_src
    .groupby("composition_id", as_index=False)
    .agg({**{c: "first" for c in other_cols}, **{c: "sum" for c in weighted_cols}})
)

if "wvp" in df_step6.columns:
    wvp = df_step6.pop("wvp")
    df_step6["wvp"] = wvp

### step7:tocsv

In [7]:
out_path = r"../../data/processed/processed_data.csv"
df_step6.to_csv(out_path, index=False)
out_path

'../../data/processed/processed_data.csv'